EinsteinPy test codes

In [ ]:
import sympy
from sympy import symbols, sin, cos, sinh, Matrix
from sympy import init_session
from einsteinpy.symbolic import EinsteinTensor, MetricTensor
# sympy.init_session # initialize all sympy contents, symbols, including init_printing
sympy.init_printing() # initializethe best printing format in the current environent



<function sympy.interactive.session.init_session(ipython=None, pretty_print=True, order=None, use_unicode=None, use_latex=None, quiet=False, auto_symbols=False, auto_int_to_Integer=False, str_printer=None, pretty_printer=None, latex_printer=None, argv=[])>

In [3]:
syms = sympy.symbols("t x y z") 
A, W = sympy.symbols("A W") 
t, x, y, z = syms

M = ([[-1,0,0,0], [0,1,0,0],[0,0,1,A*sin(W*(t-x))], [0,0,A*sin(W*(t-x)),1]])
M


In [5]:
metric = MetricTensor(M, syms)
metric.tensor() # Input - metric tensor

⎡-1  0         0                 0        ⎤
⎢                                         ⎥
⎢0   1         0                 0        ⎥
⎢                                         ⎥
⎢0   0         1          A⋅sin(W⋅(t - x))⎥
⎢                                         ⎥
⎣0   0  A⋅sin(W⋅(t - x))         1        ⎦

In [6]:
einst = EinsteinTensor.from_metric(metric) # Calculating the Einstein Tensor (with both indices covariant)

einst.tensor() # Output:  Einstein Tensor 

⎡      ⎛                                              ⎛   2    2               ↪
⎢ 2  2 ⎜  ⎛ 2    2               ⎞                    ⎝3⋅A ⋅sin (W⋅(t - x)) -  ↪
⎢A ⋅W ⋅⎜- ⎝A ⋅sin (W⋅(t - x)) - 1⎠⋅cos(2⋅W⋅(t - x)) + ──────────────────────── ↪
⎢      ⎝                                                                  2    ↪
⎢───────────────────────────────────────────────────────────────────────────── ↪
⎢                                                           2                  ↪
⎢                                   ⎛ 2    2               ⎞                   ↪
⎢                                   ⎝A ⋅sin (W⋅(t - x)) - 1⎠                   ↪
⎢                                                                              ↪
⎢      ⎛⎛     2    2               ⎞    2                                      ↪
⎢ 2  2 ⎜⎝- 3⋅A ⋅sin (W⋅(t - x)) + 1⎠⋅cos (W⋅(t - x))   ⎛ 2    2                ↪
⎢A ⋅W ⋅⎜──────────────────────────────────────────── + ⎝A ⋅sin (W⋅(t - x)) - 1 ↪
⎢      ⎝                    

Anti-de Sitter spacetime Metric

In [9]:
syms = sympy.symbols("t chi theta phi") 
syms
t, ch, th, ph = syms
m = sympy.diag(-1, cos(t) ** 2, cos(t) ** 2 * sinh(ch) ** 2, cos(t) ** 2 * sinh(ch) ** 2 * sin(th) ** 2).tolist()
metric = MetricTensor(m, syms)
metric.tensor() # Input - metric tensor
einst = EinsteinTensor.from_metric(metric) # Calculating the Einstein Tensor (with both indices covariant)
einst.tensor() # Output:  Einstein Tensor 

⎡-3.0       0                0                         0              ⎤
⎢                                                                     ⎥
⎢             2                                                       ⎥
⎢ 0    3.0⋅cos (t)           0                         0              ⎥
⎢                                                                     ⎥
⎢                          2        2                                 ⎥
⎢ 0         0       3.0⋅cos (t)⋅sinh (χ)               0              ⎥
⎢                                                                     ⎥
⎢                                                2       2        2   ⎥
⎣ 0         0                0            3.0⋅sin (θ)⋅cos (t)⋅sinh (χ)⎦

Solving the 2 coupled differential equations for the neutron stars — the "mass continuity equation and the "Tolman-Oppenheimer-Volkoff (TOV) equation" require (all in geometrized units):
1. Specify an equation of state (EOS) p(rho) relating pressure p and energy density rho.
2. Integrate numerically from the center (r = 0) to the surface (p = 0), using appropriate boundary conditions.

1. Equations to Solve
dm/dr = 4pi*r^2*rho (Mass continuity)
dp/dr = -(rho + p)(m + 4pi*r^3*p)/{r(r - 2m)} (TOV equation)

2. Boundary Conditions
At the center r = 0, m(0) = 0 (no mass enclosed at r = 0,
p(0) = p_c (central pressure, chosen based on the star's properties).
At the surface r = R: p(R) = 0 (pressure drops to zero at the surface).

3. Equation of State (EOS)
A simple but physically relevant EOS is the "polytropic equation of state":
p = K*rho^Gamma, where: K is a constant, Gamma is the adiabatic index (e.g., Gamma = 2 for neutron stars).
For a more realistic model, tabulated EOS data (e.g., SLy, APR) can be used.

4. Numerical Integration (Pseudocode) 
We use the Runge-Kutta 4th-order (RK4) method to integrate the equations outward from r = 0:
Example : for a neutron star with:
p_c = 10^35 Pa, rh_c = 5 * 10^17 kg/m^3, Gamma = 2, K = 1 * 10^5
You would obtain:
Mass M approx = 1.4 solar masses,
Radius R approx = 10 km.

Iteration 0

In [23]:
import numpy as np
from sympy import *
Gamma = 2
K = 1*10**5
r0 = 0.01
rho_0 = (5*10**17)*(7.42592*10**(-28)) # convert into geometrized unit
print(f"rho = {rho_0}")
print(f"r = {r0}")
m0 = 4/3 * np.pi * r0**3 * rho_0
print("m = {:3e}".format(m0))
m0
print("r - 2m ={:3e}".format(r0 - 2*m0))
p0 = K * rho_0 ** Gamma
print("p = {:3e}".format(p0))
p0
dp0_dr = - (rho_0 + p0) * (m0 + 4 * np.pi * r0**3 * p0) / (r0 * (r0 - 2 * m0) )
print("dp/dr ={:3e}".format(dp0_dr))
dp0_dr

rho = 3.7129599999999997e-10
r = 0.01
m = 1.555281e-15
r - 2m =1.000000e-02
p = 1.378607e-14
dp/dr =-5.775554e-21


Iteration 1

In [ ]:
r1 = r0 + 1
p1 = p0 + dp0_dr * 1
rho_1 = (p1 / K )**(1/Gamma)
print(f"rho = {rho_1}")
print(f"r = {r1}")
m1 = 4/3 * np.pi * r1**3 * rho_1
print("m = {:3e}".format(m1))
print("r - 2m ={:3e}".format(r1- 2*m1))
print("p = {:3e}".format(p1))

dp1_dr = - (rho_1 + p1) * (m1 + 4 * np.pi * r1**3 * p1) / (r1 * (r1 - 2 * m1) )
print("dp/dr ={:3e}".format(dp1_dr))
dp1_dr


rho = 3.7129592222438976e-10
r = 1.01
m = 1.602407e-09
r - 2m =1.010000e+00
p = 1.378607e-14
dp/dr =-5.833307e-19


Iteration 2

In [26]:
r2 = r1 + 1
p2 = p1 + dp1_dr * 1
rho_2 = (p2 / K )**(1/Gamma)
print(f"rho = {rho_2}")
print(f"r = {r2}")
m2 = 4/3 * np.pi * r2**3 * rho_2
print("m = {:3e}".format(m2))
print("r - 2m ={:3e}".format(r2- 2*m2))
print("p = {:3e}".format(p2))

dp2_dr = - (rho_2 + p2) * (m2 + 4 * np.pi * r2**3 * p2) / (r2 * (r2 - 2 * m2) )
print("dp/dr ={:3e}".format(dp2_dr))
dp2_dr

rho = 3.712880668071e-10
r = 2.01
m = 1.262955e-08
r - 2m =2.010000e+00
p = 1.378548e-14
dp/dr =-1.160837e-18


Iteration 3

In [28]:
r3 = r2 + 1
p3 = p2 + dp2_dr * 1
rho_3 = (p3 / K )**(1/Gamma)
print(f"rho = {rho_3}")
print(f"r = {r3}")
m3 = 4/3 * np.pi * r3**3 * rho_3
print("m = {:3e}".format(m3))
print("r - 2m ={:3e}".format(r3- 2*m3))
print("p = {:3e}".format(p3))

dp3_dr = - (rho_3 + p3) * (m3 + 4 * np.pi * r3**3 * p3) / (r3 * (r3 - 2 * m3) )
print("dp/dr ={:3e}".format(dp3_dr))
dp3_dr

rho = 3.71272433915838e-10
r = 3.01
m = 4.241122e-08
r - 2m =3.010000e+00
p = 1.378432e-14
dp/dr =-1.738221e-18


Not pretty", pure text... "low quality". This is what one obtains with print(expression)
Pretty, pure text... "medium quality". This is what one obtains with

 import sympy as sym
 sympy.pprint(expression)
It still uses the same font and uses only characters to put together the mathematical expression. But it can, e.g., raise numbers for powers, pull fractions by laying out the horizontal line, etc.

Pretty, with graphics, symbols, etc... "high quality". This is what one obtains with

 import IPython.display as disp
 disp.display(expression)
This is the same as what one obtains as an output of the notebook cell, but now as a result of a command. Then, one can have multiple such outputs from a single notebook cell.

It is worth noting that:

sym.init_printing(... affects the output of sym.pprint.
sym.latex(expression) produces a LaTeX string for the expression. disp.Math(... produces the expression from LaTeX. These two may come in useful. Thus, disp.display(disp.Math(sym.latex(expression))) would produce the same output as disp.display(expression).

dp/dr =-5.775554e-21


In [ ]:
import numpy as np

def TOV_solver(p_c, rho_c, Gamma, K, r_max, dr):
    # Initialize
    r = 0
    m = 0
    p = p_c
    rho = rho_c
    
    # Arrays to store results
    r_list = [r]
    m_list = [m]
    p_list = [p]
    rho_list = [rho]
    
    # Integration loop
    
    while r < r_max and p > 0:
        # Compute derivatives
        dm_dr = 4 * np.pi * r**2 * rho
        dp_dr = - (rho + p) * (m + 4 * np.pi * r**3 * p) / (r * (r - 2 * m) + 1e-10) # Avoid division by zero
        
        # RK4 integration (Implement RK4 steps here)
        
        # Update variables
        m += dm_dr * dr
        p += dp_dr * dr
        r += dr
        
        # Update density using EOS
        rho = (p / K)**(1 / Gamma)
        
        # Store results
        r_list.append(r)
        m_list.append(m)
        p_list.append(p)
        rho_list.append(rho)
    
    return r_list, m_list, p_list, rho_list

TOV_solver(p_c = 10**35, rho_c = 5*10**17, Gamma = 2, K = 1*10**5, r_max=50, dr=0.01)


5. Results
The integration stops when p drops to zero, defining the star's radius R.
The total gravitational mass is M = m(R).
The **density profile rho(r)  and pressure profile p(r) describe the star's internal structure.

Key Notes
Relativistic effects: The TOV equation deviates from Newtonian gravity when m/r is large (e.g., near neutron star cores).
Stability: Solutions are valid only if M and R satisfy stability criteria.
More realistic models: Use tabulated EOS data for precise neutron star predictions.